# Transformer 정답 — N-1–N-36

먼저 [빈칸 노트북](A.ing_Transformer_from_scratch_blank.ipynb)을 채운 뒤 확인하세요.
이 파일은 **실행 가능한 완성 모델과 동일한 핵심 검사·비교 실험**을 제공합니다. PyTorch만 있으면 위에서부터 실행할 수 있습니다.
Multi30k 설치·학습 코드는 빈칸 노트북의 선택 실습을 사용하세요. 근거는 [논문 가이드](../Week1/transformer_paper_guide.md)와 [쿡북](A.ing_Transformer_Cookbook.md)에서 확인하세요.

기반 코드: Aladdin Persson, A.ing 교육용 수정. [라이선스](../README.md).


In [ ]:
import torch
import torch.nn as nn


In [ ]:
def study_check_shape(actual, expected, where, route):
    actual, expected = tuple(actual), tuple(expected)
    if actual != expected:
        differences = [f"축 {i}: {a} != {b}" for i, (a, b) in enumerate(zip(actual, expected)) if a != b]
        if len(actual) != len(expected):
            differences.append(f"차원 수 {len(actual)} != {len(expected)}")
        raise AssertionError(f"{where}: 기대 {expected}, 실제 {actual}. " + "; ".join(differences) + f" → {route}")
    print(f"[OK] {where}: {actual}")


def study_check(condition, message, route):
    if not bool(condition):
        raise AssertionError(f"{message} → {route}")


def study_small_model(seed=7, layers=1):
    # 진단용 모델. 학습 데이터나 GPU 없이 실행합니다.
    torch.manual_seed(seed)
    return Transformer(24, 24, 0, 0, embed_size=32, num_layers=layers,
                       forward_expansion=2, heads=4, dropout=0.0,
                       device="cpu", max_length=16)


In [ ]:
# =========================================================
# 1) SelfAttention (Scaled Dot-Product + Multi-Head)
# - CheatSheet §1~2 | CookBook step.2, 6~12
# - Eq.(1): softmax(QK^T / sqrt(d_k)) V
# =========================================================

class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.d_model = embed_size
        self.h = heads
        # [CheatSheet §2 Multi-Head Attention | Step 1/2]
        # 힌트: A.ing_Transformer_Cookbook.md [step.2]의 입력/출력 shape을 먼저 확인하세요.
        self.d_k = self.d_model // self.h  # [N-1]
        # [CheatSheet §2 Multi-Head Attention | Step 2/2]
        assert (
            self.h * self.d_k == self.d_model
        ), "Embedding size needs to be divisible by heads"

        # [CheatSheet §1 Scaled Dot-Product Attention | Step 1/2] Q,K,V 선형변환 정의 (Eq.1)
        # 힌트: A.ing_Transformer_Cookbook.md [step.6]의 입력/출력 shape을 먼저 확인하세요.
        # ================================================================================
        # Multi Head Self Attention:  원래는 Q, K, V를 (d_model -> d_k) 크기로 변환하는 작업입니다.
        # 하지만 논문의 구조를 그대로 따르려면 이 연산을 8번 해야하는데, 이러면 연산이 너무 비효율적입니다.
        # 그렇다면 원래 사이즈로 변환을 한 다음 그것을 head의 수만큼 나눠주면 되겠죠?
        # 이 말이 이해 되셨다면 빈칸을 채우실 수 있습니다!
        # ================================================================================
        self.W_V = nn.Linear(self.d_model, self.d_model)  # [N-2]
        self.W_K = nn.Linear(self.d_model, self.d_model)  # [N-3]
        self.W_Q = nn.Linear(self.d_model, self.d_model)  # [N-4]
        # [CheatSheet §2 Multi-Head Attention | Step 2/2] Concat(heads) 이후 W^O (Fig.2)
        # 힌트: A.ing_Transformer_Cookbook.md [step.12]의 입력/출력 shape을 먼저 확인하세요.
        self.W_O = nn.Linear(self.d_model, self.d_model)  # [N-5]

    def forward(self, values, keys, query, mask):
        # [CheatSheet Shape 규칙 | Step 1/9] 배치 크기 N / 길이 추출
        # ================================================================================
        # NLP 모델에서 텐서의 shape은 보통 (배치 크기, 문장 길이, 임베딩 차원) 순서로 들어옵니다.
        # CheatSheet Shape 규칙을 보셨다면 N이 무엇을 의미하는지 아실 겁니다.
        # ================================================================================
        N = query.shape[0]  # [N-6]

        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        # [CheatSheet §1 Scaled Dot-Product Attention | Step 2/9] Q, K, V 만들기 (Eq.1)
        # 힌트: A.ing_Transformer_Cookbook.md [step.6]의 입력/출력 shape을 먼저 확인하세요.
        V = self.W_V(values)
        K = self.W_K(keys)
        Q = self.W_Q(query)

        # [CheatSheet §2 Multi-Head Attention | Step 3/9]
        # 힌트: A.ing_Transformer_Cookbook.md [step.7]의 입력/출력 shape을 먼저 확인하세요.
        # ================================================================================
        # 여기서는 Multi Head Attention을 위해 Q, K, V를 head 수만큼 나누는 과정입니다.
        # 기존의 Q, K, V의 shape은 (배치 크기(N), 문장 길이(len), d_model)입니다.
        # 여기서 head와 관련된 연산이 수행된 변수가 하나 있습니다.
        # 이제 빈칸을 풀어보세요!
        # ================================================================================
        V = V.reshape(N, value_len, self.h, self.d_k)  # [N-7]
        K = K.reshape(N, key_len, self.h, self.d_k)  # [N-8]
        Q = Q.reshape(N, query_len, self.h, self.d_k)  # [N-9]

        # [CheatSheet §1 Scaled Dot-Product Attention | Step 4/9] attention_logits = QK^T (Eq.1)
        # 힌트: A.ing_Transformer_Cookbook.md [step.8]의 입력/출력 shape을 먼저 확인하세요.
        attention_logits_QK = torch.einsum("nqhd,nkhd->nhqk", [Q, K])  # [N-10]

        # [CheatSheet §4 Mask | Step 5/9] mask 적용 (padding/causal 차단)
        # 힌트: A.ing_Transformer_Cookbook.md [step.9]의 입력/출력 shape을 먼저 확인하세요.
        if mask is not None:
            attention_logits_QK = attention_logits_QK.masked_fill(mask == 0, float("-1e20"))  # [N-11]

        # [CheatSheet §1 Scaled Dot-Product Attention | Step 6/9] scale(+softmax) (Eq.1)
        # - / sqrt(d_k) 로 softmax 포화 방지
        # - softmax dim은 key_len 축(마지막 축)
        # 힌트: A.ing_Transformer_Cookbook.md [step.10]의 입력/출력 shape을 먼저 확인하세요.
        attention_weights = torch.softmax(attention_logits_QK / (self.d_k ** (1 / 2)), dim=-1)

        # [CheatSheet §1 Scaled Dot-Product Attention | Step 7/9] out_heads = attention_weights @ V (Eq.1)
        # 힌트: A.ing_Transformer_Cookbook.md [step.11]의 입력/출력 shape을 먼저 확인하세요.
        out_heads = torch.einsum("nhql,nlhd->nqhd", [attention_weights, V])  # [N-12]
        # [CheatSheet §2 Multi-Head Attention | Step 8/9]
        out = out_heads.reshape(N, query_len, self.h * self.d_k)  # [N-13]

        # [CheatSheet §2 Multi-Head Attention | Step 9/9] W^O output projection (Fig.2)
        out = self.W_O(out)

        return out


In [ ]:
import torch

torch.manual_seed(0)

N = 2
q_len = 5
d_model = 32
h = 4

self_attn = SelfAttention(embed_size=d_model, heads=h)

query = torch.randn(N, q_len, d_model)  # (N, q_len, d_model)
keys = query  # (N, q_len, d_model)
values = query  # (N, q_len, d_model)

mask = torch.ones(N, 1, 1, q_len)  # (N, 1, 1, k_len)
mask[0, :, :, -1] = 0  # (N, 1, 1, k_len)

out = self_attn(values=values, keys=keys, query=query, mask=mask)  # (N, q_len, d_model)
study_check_shape(out.shape, (N, q_len, d_model), "out", "C7–C12 / N-7–N-13")
assert torch.isfinite(out).all(), "NaN/Inf detected in SelfAttention output"
print("[OK] SelfAttention output shape:", tuple(out.shape))


In [ ]:
import torch.nn.functional as F

torch.manual_seed(11)
attn = SelfAttention(embed_size=32, heads=4).eval()
q = torch.randn(2, 3, 32)
k = torch.randn(2, 5, 32)
v = torch.randn(2, 5, 32)
visible = torch.ones(2, 1, 1, 5, dtype=torch.bool)
visible[..., -1] = False
with torch.no_grad():
    actual = attn(v, k, q, visible)
    study_check_shape(actual.shape, (2, 3, 32), "cross-attention", "C12/C17 / N-13/N-22")
    k_changed, v_changed = k.clone(), v.clone()
    k_changed[:, -1] += 100
    v_changed[:, -1] -= 100
    masked_change = attn(v_changed, k_changed, q, visible)
    study_check(torch.allclose(actual, masked_change, atol=1e-5),
                "차단한 key/value의 내용이 출력에 섞였습니다", "C3/C9 / N-11")
    active_v = v.clone()
    active_v[:, 0] += 10
    study_check(not torch.allclose(actual, attn(active_v, k, q, visible), atol=1e-5),
                "허용한 value를 바꿔도 출력이 같습니다", "C11/C12 / N-12/N-13")
    # 별도 API를 기준으로 projection부터 head 결합까지 비교합니다.
    def split_reference(layer, x):
        return layer(x).unflatten(-1, (4, 8)).transpose(1, 2)
    expected = F.scaled_dot_product_attention(
        split_reference(attn.W_Q, q), split_reference(attn.W_K, k),
        split_reference(attn.W_V, v), attn_mask=visible, dropout_p=0.0)
    expected = attn.W_O(expected.transpose(1, 2).flatten(-2))
    study_check(torch.allclose(actual, expected, atol=1e-5, rtol=1e-4),
                "독립 attention 기준과 값이 다릅니다", "C6–C12 / N-2–N-13")
print("[OK] Cross-attention 값·길이·padding 검사")


In [ ]:
# =========================================================
# 5) TransformerBlock (Add & Norm + FFN)
# - CheatSheet §5 | CookBook step.13~14
# =========================================================

class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()
        self.attention = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        # [CheatSheet §5 Add & Norm + FFN | Step 1/2]
        # 힌트: A.ing_Transformer_Cookbook.md [step.14]의 입력/출력 shape을 먼저 확인하세요.
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        # [CheatSheet §5 Add & Norm + FFN | Step 1/4] (Sublayer) Multi-Head Attention
        # 힌트: A.ing_Transformer_Cookbook.md [step.13]의 입력/출력 shape을 먼저 확인하세요.
        multihead_attention_output = self.attention(value, key, query, mask)

        # [CheatSheet §5 Add & Norm + FFN | Step 2/4] Add & Norm (Residual + LayerNorm + Dropout)
        # ================================================================================
        # Residual은 입력을 더해 정보가 흐를 통로를 만듭니다.
        # 여기서는 원논문 §3.1의 Add & Norm 순서와 덧셈 shape을 확인하세요.
        # 현재 단계는 query에 대해 가장 관련있는 value를 찾는 과정이면 attention 결과에 무엇을 더해야 할지 아시겠죠?
        # ================================================================================
        attention_residual_add = multihead_attention_output + query  # [N-14]
        post_attention_layernorm = self.norm1(attention_residual_add)
        x = self.dropout(post_attention_layernorm)

        # [CheatSheet §5 Add & Norm + FFN | Step 3/4] Position-wise FFN (shape를 보이게 쪼개서 실행)
        # 힌트: A.ing_Transformer_Cookbook.md [step.14]의 입력/출력 shape을 먼저 확인하세요.
        ffn_linear1_output = self.feed_forward[0](x)
        ffn_relu_output = self.feed_forward[1](ffn_linear1_output)
        ffn_linear2_output = self.feed_forward[2](ffn_relu_output)

        # [CheatSheet §5 Add & Norm + FFN | Step 4/4] Add & Norm after FFN
        # ================================================================================
        # Residual은 입력을 더해 정보가 흐를 통로를 만듭니다.
        # 여기서는 원논문 §3.1의 Add & Norm 순서와 덧셈 shape을 확인하세요.
        # 현재 단계는 x에 대해 FFN 결과를 출력하는 단계이니 skip connection을 구상할 아이디어가 떠오르시죠?
        # ================================================================================
        ffn_residual_add = ffn_linear2_output + x  # [N-15]
        post_ffn_layernorm = self.norm2(ffn_residual_add)
        out = self.dropout(post_ffn_layernorm)

        return out


In [ ]:
import torch

torch.manual_seed(0)

N = 2
seq_len = 6
d_model = 32
h = 4

block = TransformerBlock(embed_size=d_model, heads=h, dropout=0.0, forward_expansion=4)

x = torch.randn(N, seq_len, d_model)  # (N, seq_len, d_model)
mask = torch.ones(N, 1, 1, seq_len)  # (N, 1, 1, seq_len)

out = block(value=x, key=x, query=x, mask=mask)  # (N, seq_len, d_model)
study_check_shape(out.shape, (N, seq_len, d_model), "out", "C13–C14 / N-14–N-15")
assert torch.isfinite(out).all(), "NaN/Inf detected in TransformerBlock output"
print("[OK] TransformerBlock output shape:", tuple(out.shape))


In [ ]:
# =========================================================
# 6) Encoder (Embedding + Positional Encoding + Encoder Stack)
# - CheatSheet §6, §3(Encoder Self-Attention) | CookBook step.1, 4~5, 15
# =========================================================

class Encoder(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        embed_size,
        num_layers,
        heads,
        device,
        forward_expansion,
        dropout,
        max_length,
    ):

        super(Encoder, self).__init__()
        self.embed_size = embed_size
        self.device = device
        self.word_embedding = nn.Embedding(src_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)

        self.layers = nn.ModuleList(
            [
                TransformerBlock(
                    embed_size,
                    heads,
                    dropout=dropout,
                    forward_expansion=forward_expansion,
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, src_token_ids, src_padding_mask):
        # [CheatSheet §0 Shape 규칙 | Step 1/3] src_token_ids: (N, src_len)
        # 힌트: A.ing_Transformer_Cookbook.md [step.1]의 입력/출력 shape을 먼저 확인하세요.
        N, src_len = src_token_ids.shape

        # [CheatSheet §6 Embedding + Positional Encoding | Step 2/3] positions 만들기
        # 힌트: A.ing_Transformer_Cookbook.md [step.5]의 입력/출력 shape을 먼저 확인하세요.
        # ================================================================================
        # Positional Encoding을 알고 있다면 "위치 인덱스 만들기:"부터 읽으셔도 됩니다.
        # RNN은 본질적으로 단어가 순서대로 들어오기 때문에 장기의존성 문제가 생깁니다. 이 문제를 해결하기 위해 Transformer는 모든 단어를 한 번에 병렬연산합니다.
        # 위치 정보가 없는 self-attention은 입력 순서를 바꾸면 출력도 같은 순서로 바뀝니다. E02에서 이 성질을 확인합니다. 따라서 문장의 "토큰"마다 고유한 위치정보를 더해야 합니다.

        # 위치 인덱스 만들기:
        # 1. arange(0, x)를 사용해 [0, 1, 2, ..., "문장길이"-1] 형태의 1차원 번호표를 만듭니다.
        # 2. expand(N, L)는 같은 위치 번호를 N개 문장에 broadcast합니다. 실제 데이터를 복사하지 않습니다.
        # 즉, (L,)을 (N, L)로 확장합니다.
        # 그렇다면 빈칸을 어떻게 채워야 할까요?
        # ================================================================================

        position_ids = torch.arange(0, src_len).to(self.device)  # [N-16]
        positions = position_ids.expand(N, src_len)  # [N-17]

        # [CheatSheet §6 Embedding + Positional Encoding | Step 3/3] token_emb + pos_emb (+ dropout)
        # 힌트: A.ing_Transformer_Cookbook.md [step.4~5]
        token_embedding_d_model = self.word_embedding(src_token_ids)
        positional_embedding_d_model = self.position_embedding(positions)
        out = self.dropout(token_embedding_d_model + positional_embedding_d_model)  # [N-18]


        # [CheatSheet §3 Encoder Self-Attention | Step 1/1] Encoder stack 반복
        # 힌트: A.ing_Transformer_Cookbook.md [step.15]의 입력/출력 shape을 먼저 확인하세요.
        # ================================================================================
        # 여기서 layer는 self.layers의 TransformerBlock 객체입니다.
        # 아마 많은 분들이 놓치실텐데, torch의 객체를 불러오는 것만으로도 객체 내의 forward 함수는 실행됩니다.
        # 위의 두 문장을 연결해서 생각해보세요. 그러면 아래 4개의 입력은 Transformer의 어떤 함수를 실행시키기 위해 필요할까요?
        # 질문에 답을 할 수 있다면 빈칸을 채우실 수 있습니다!
        # ================================================================================
        for layer in self.layers:
            out = layer(out, out, out, src_padding_mask)  # [N-19]

        return out


In [ ]:
import torch

torch.manual_seed(0)

N = 2
src_len = 7
src_vocab_size = 50
src_pad_idx = 0
d_model = 32
h = 4

device = torch.device("cpu")

encoder = Encoder(
    src_vocab_size=src_vocab_size,
    embed_size=d_model,
    num_layers=2,
    heads=h,
    device=device,
    forward_expansion=4,
    dropout=0.0,
    max_length=100,
).to(device)

src_token_ids = torch.randint(1, src_vocab_size, (N, src_len)).to(device)  # (N, src_len)
src_token_ids[0, -2:] = src_pad_idx  # (N, src_len)

src_padding_mask = (src_token_ids != src_pad_idx).unsqueeze(1).unsqueeze(2)  # (N, 1, 1, src_len)

enc_out = encoder(src_token_ids=src_token_ids, src_padding_mask=src_padding_mask)  # (N, src_len, d_model)
study_check_shape(enc_out.shape, (N, src_len, d_model), "enc_out", "C5/C15 / N-16–N-19")
assert torch.isfinite(enc_out).all(), "NaN/Inf detected in Encoder output"
print("[OK] Encoder output shape:", tuple(enc_out.shape))


In [ ]:
# =========================================================
# 3) DecoderBlock (Masked Self-Attention + Cross-Attention)
# - CheatSheet §3 | CookBook step.16~17
# =========================================================

class DecoderBlock(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion, dropout, device):
        super(DecoderBlock, self).__init__()
        self.norm = nn.LayerNorm(embed_size)
        self.attention = SelfAttention(embed_size, heads=heads)
        self.transformer_block = TransformerBlock(
            embed_size, heads, dropout, forward_expansion
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, value, key, src_mask, trg_mask):
        # [CheatSheet §3 Decoder Masked Self-Attention | Step 1/3] masked self-attn (미래 토큰 차단)
        # 힌트: A.ing_Transformer_Cookbook.md [step.16]의 입력/출력 shape을 먼저 확인하세요.
        # ================================================================================
        # 코드 내에서 self.attention은 SelfAttention 객체이니 이 객체를 호출했다면 forward 함수를 실행해야 합니다.
        # 그리고 여기는 Decoder이므로 Masked Multi Head Self Attention을 해야 합니다.
        # 그러면 어디를 찾아야 할지 아시겠죠??????????
        # ================================================================================

        masked_self_attention_output = self.attention(x, x, x, trg_mask)  # [N-20]

        # [CheatSheet §5 Add & Norm | Step 2/3] Residual + LayerNorm + Dropout (query 만들기)
        residual_add = masked_self_attention_output + x  # [N-21]
        query = self.dropout(self.norm(residual_add))

        # [CheatSheet §3 Encoder-Decoder Attention | Step 3/3] Cross-Attention(+FFN) via TransformerBlock
        # 힌트: A.ing_Transformer_Cookbook.md [step.17]의 입력/출력 shape을 먼저 확인하세요.
        out = self.transformer_block(value, key, query, src_mask)  # [N-22]

        return out


In [ ]:
import torch

torch.manual_seed(0)

N = 2
src_len = 7
trg_len = 5
d_model = 32
h = 4

device = torch.device("cpu")

decoder_block = DecoderBlock(
    embed_size=d_model,
    heads=h,
    forward_expansion=4,
    dropout=0.0,
    device=device,
).to(device)

x = torch.randn(N, trg_len, d_model).to(device)  # (N, trg_len, d_model)
value = torch.randn(N, src_len, d_model).to(device)  # (N, src_len, d_model)
key = value  # (N, src_len, d_model)

src_mask = torch.ones(N, 1, 1, src_len).to(device)  # (N, 1, 1, src_len)

trg_mask_base = torch.tril(torch.ones(trg_len, trg_len)).to(device)  # (trg_len, trg_len)
trg_mask = trg_mask_base.expand(N, 1, trg_len, trg_len)  # (N, 1, trg_len, trg_len)

out = decoder_block(x=x, value=value, key=key, src_mask=src_mask, trg_mask=trg_mask)  # (N, trg_len, d_model)
study_check_shape(out.shape, (N, trg_len, d_model), "out", "C16/C17 / N-20–N-22")
assert torch.isfinite(out).all(), "NaN/Inf detected in DecoderBlock output"
print("[OK] DecoderBlock output shape:", tuple(out.shape))


In [ ]:
# =========================================================
# 6) Decoder (Embedding + Positional Encoding + Decoder Stack + Vocab Projection)
# - CheatSheet §6, §7 | CookBook step.1, 4~5, 18
# =========================================================

class Decoder(nn.Module):
    def __init__(
        self,
        trg_vocab_size,
        embed_size,
        num_layers,
        heads,
        forward_expansion,
        dropout,
        device,
        max_length,
    ):
        super(Decoder, self).__init__()
        self.device = device
        self.word_embedding = nn.Embedding(trg_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)

        self.layers = nn.ModuleList(
            [
                DecoderBlock(embed_size, heads, forward_expansion, dropout, device)
                for _ in range(num_layers)
            ]
        )
        self.fc_out = nn.Linear(embed_size, trg_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, trg_token_ids, enc_out, src_padding_mask, trg_causal_mask):
        # [CheatSheet §0 Shape 규칙 | Step 1/4] trg_token_ids: (N, trg_len)
        # 힌트: A.ing_Transformer_Cookbook.md [step.1]의 입력/출력 shape을 먼저 확인하세요.
        N, trg_len = trg_token_ids.shape

        # [CheatSheet §6 Embedding + Positional Encoding | Step 2/4] positions 만들기
        # 힌트: A.ing_Transformer_Cookbook.md [step.5]의 입력/출력 shape을 먼저 확인하세요.
        position_ids = torch.arange(0, trg_len).to(self.device)  # [N-23]
        positions = position_ids.expand(N, trg_len)  # [N-24]

        # [CheatSheet §6 Embedding + Positional Encoding | Step 3/4] token_emb + pos_emb (+ dropout)
        token_embedding_d_model = self.word_embedding(trg_token_ids)
        positional_embedding_d_model = self.position_embedding(positions)
        x = self.dropout(token_embedding_d_model + positional_embedding_d_model)  # [N-25]

        # [CheatSheet §3 Decoder Stack | Step 1/1] DecoderBlock 반복
        # 힌트: A.ing_Transformer_Cookbook.md [step.18]의 입력/출력 shape을 먼저 확인하세요.
        for layer in self.layers:
            x = layer(x, enc_out, enc_out, src_padding_mask, trg_causal_mask)  # [N-26]

        # [CheatSheet §7 Output projection | Step 4/4] vocab logits 생성
        # 힌트: A.ing_Transformer_Cookbook.md [step.18]의 입력/출력 shape을 먼저 확인하세요.
        out = self.fc_out(x)  # [N-27]

        return out


In [ ]:
# =========================================================
# 0) Transformer (Encoder + Decoder + Mask 2종)
# - CheatSheet §0(전체), §4(Mask) | CookBook step.3, 19
# =========================================================
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        trg_vocab_size,
        src_pad_idx,
        trg_pad_idx,
        embed_size=512,
        num_layers=6,
        forward_expansion=4,
        heads=8,
        dropout=0,
        device="cpu",
        max_length=100,
    ):

        super(Transformer, self).__init__()

        self.encoder = Encoder(
            src_vocab_size,
            embed_size,
            num_layers,
            heads,
            device,
            forward_expansion,
            dropout,
            max_length,
        )

        self.decoder = Decoder(
            trg_vocab_size,
            embed_size,
            num_layers,
            heads,
            forward_expansion,
            dropout,
            device,
            max_length,
        )

        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

    def make_src_mask(self, src_token_ids):
        # [CheatSheet §4 Mask | Step 1/2] src padding mask: (src != pad) -> unsqueeze(1)->unsqueeze(2)
        # 힌트: A.ing_Transformer_Cookbook.md [step.3]의 입력/출력 shape을 먼저 확인하세요.
        # ================================================================================
        # 이거 테스트로 풀어볼 때 많이 어려웠습니다ㅜㅜ. 일단 변수 이름부터 읽어볼까요?
        # "src_is_not_pad"는 src(원본)에서 is_not_pad(패딩이 아닌 부분)입니다. 그렇다면 src_token과 "무엇"이 같지 않은 부분을 확인해야 할까요?
        # ================================================================================
        src_is_not_pad = (src_token_ids != self.src_pad_idx)  # [N-28]
        src_padding_mask = src_is_not_pad.unsqueeze(1)  # [N-29]
        src_padding_mask = src_padding_mask.unsqueeze(2)  # [N-30]
        return src_padding_mask.to(self.device)

    def make_trg_mask(self, trg_token_ids):
        # [CheatSheet §4 Mask | Step 2/2] trg causal mask: tril(ones(L,L)).expand(N, 1, L, L)
        # 힌트: A.ing_Transformer_Cookbook.md [step.3]의 입력/출력 shape을 먼저 확인하세요.
        N, trg_len = trg_token_ids.shape
        trg_ones = torch.ones((trg_len, trg_len))  # [N-31]
        trg_lower_triangular = torch.tril(trg_ones)  # 설명: 하삼각(미래 토큰 차단)
        trg_causal_mask = trg_lower_triangular.expand(N, 1, trg_len, trg_len)  # [N-32]

        return trg_causal_mask.to(self.device)

    def forward(self, src_token_ids, trg_token_ids):
        # [CheatSheet 전체 구조 | Step 1/3] mask 만들기
        # 힌트: A.ing_Transformer_Cookbook.md [step.19]의 입력/출력 shape을 먼저 확인하세요.
        src_padding_mask = self.make_src_mask(src_token_ids)  # [N-33]
        trg_causal_mask = self.make_trg_mask(trg_token_ids)  # [N-34]

        # [CheatSheet 전체 구조 | Step 2/3] Encoder
        # ================================================================================
        # 객체를 호출하면 객체 안의 어떤 함수가 자동으로 호출된다고 정말 많이 남겨뒀습니다.
        # 그러면 이제 찾아봅시다!
        # ================================================================================
        enc_src = self.encoder(src_token_ids, src_padding_mask)  # [N-35]
        # [CheatSheet 전체 구조 | Step 3/3] Decoder
        out = self.decoder(trg_token_ids, enc_src, src_padding_mask, trg_causal_mask)  # [N-36]

        return out


In [ ]:
mask_only = Transformer.__new__(Transformer)
nn.Module.__init__(mask_only)
mask_only.src_pad_idx = 0
mask_only.trg_pad_idx = 0
mask_only.device = "cpu"
src_demo = torch.tensor([[1, 4, 0], [1, 5, 2]])
trg_demo = torch.tensor([[1, 4, 2], [1, 6, 2]])
src_mask_demo = mask_only.make_src_mask(src_demo)
trg_mask_demo = mask_only.make_trg_mask(trg_demo)
study_check_shape(src_mask_demo.shape, (2, 1, 1, 3), "source mask", "C3 / N-28–N-30")
study_check_shape(trg_mask_demo.shape, (2, 1, 3, 3), "target mask", "C3 / N-31–N-32")
study_check(src_mask_demo[0, 0, 0].tolist() == [True, True, False],
            "source PAD 위치를 잘못 가렸습니다", "C3 / N-28–N-30")
for row in range(3):
    for col in range(3):
        study_check(bool(trg_mask_demo[0, 0, row, col]) == (col <= row),
                    f"causal mask ({row}, {col}) 위치가 잘못됐습니다", "C3 / N-31–N-32")
print("[OK] Mask만 먼저 확인했습니다")


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

N = 2
src_len = 8
trg_len = 6

src_vocab_size = 100
trg_vocab_size = 120

src_pad_idx = 0
trg_pad_idx = 0

device = torch.device("cpu")

model = Transformer(
    src_vocab_size=src_vocab_size,
    trg_vocab_size=trg_vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embed_size=32,
    num_layers=2,
    forward_expansion=4,
    heads=4,
    dropout=0.0,
    device=device,
    max_length=100,
).to(device)

src_token_ids = torch.randint(1, src_vocab_size, (N, src_len)).to(device)  # (N, src_len)
trg_token_ids = torch.randint(1, trg_vocab_size, (N, trg_len)).to(device)  # (N, trg_len)

src_token_ids[0, -2:] = src_pad_idx  # (N, src_len)
trg_token_ids[0, -1] = trg_pad_idx  # (N, trg_len)

trg_input_ids = trg_token_ids[:, :-1]  # (N, trg_len-1)
trg_target_ids = trg_token_ids[:, 1:]  # (N, trg_len-1)

src_padding_mask = model.make_src_mask(src_token_ids)  # (N, 1, 1, src_len)
trg_causal_mask = model.make_trg_mask(trg_input_ids)  # (N, 1, trg_len-1, trg_len-1)

study_check_shape(src_padding_mask.shape, (N, 1, 1, src_len), "src_padding_mask", "C3/C19 / N-28–N-36")
study_check_shape(trg_causal_mask.shape, (N, 1, trg_len - 1, trg_len - 1), "trg_causal_mask", "C3/C19 / N-28–N-36")

logits = model(src_token_ids, trg_input_ids)  # (N, trg_len-1, trg_vocab_size)
study_check_shape(logits.shape, (N, trg_len - 1, trg_vocab_size), "logits", "C3/C19 / N-28–N-36")
assert torch.isfinite(logits).all(), "NaN/Inf detected in Transformer logits"

criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)

logits_2d = logits.reshape(-1, trg_vocab_size)  # (N*(trg_len-1), trg_vocab_size)
targets_1d = trg_target_ids.reshape(-1)  # (N*(trg_len-1),)

loss = criterion(logits_2d, targets_1d)
loss.backward()

print("[OK] Transformer forward + loss backward:", loss.detach().item())


In [ ]:
m = study_small_model(layers=2).eval()
src = torch.tensor([[1, 4, 5, 2], [1, 7, 8, 2]])
target = torch.tensor([[1, 9, 10, 2], [1, 11, 12, 2]])
changed_target = target.clone()
changed_target[:, 2:] = torch.tensor([[15, 16], [17, 18]])
with torch.no_grad():
    base = m(src, target)
    altered = m(src, changed_target)
    study_check(torch.allclose(base[:, :2], altered[:, :2], atol=1e-5),
                "미래 target이 이전 logits에 영향을 줍니다", "C3/C16/C19 / N-20/N-31–N-36")
    padded_src = torch.nn.functional.pad(src, (0, 2), value=0)
    study_check(torch.allclose(base, m(padded_src, target), atol=1e-5),
                "source 뒤 PAD를 추가하자 logits가 바뀌었습니다", "C3/C15/C17 / N-19/N-22/N-28–N-30")
    # 원인이 mask임을 확인하는 대조군: 진단용 복사본만 미래를 허용합니다.
    import copy
    no_causal = copy.deepcopy(m)
    no_causal.make_trg_mask = lambda t: torch.ones(t.size(0), 1, t.size(1), t.size(1), dtype=torch.bool)
    leak = (no_causal(src, target)[:, :2] - no_causal(src, changed_target)[:, :2]).abs().max().item()
    study_check(leak > 1e-5, "대조군에서도 변화가 없어 검사가 구분력을 갖지 못합니다", "C16/C17 / N-20–N-22")
print(f"[OK] 미래 차단 / source PAD 불변성. 미래 허용 대조군 최대 차이={leak:.6f}")


In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)

    x = torch.tensor([[1, 5, 6, 4, 3, 9, 5, 2, 0], [1, 8, 7, 3, 4, 5, 6, 7, 2]]).to(
        device
    )
    # (N, src_len)
    trg = torch.tensor([[1, 7, 4, 3, 5, 9, 2, 0], [1, 5, 6, 2, 4, 7, 6, 2]]).to(device)
    # (N, trg_len)

    src_pad_idx = 0
    trg_pad_idx = 0
    src_vocab_size = 10
    trg_vocab_size = 10
    model = Transformer(
        src_vocab_size,
        trg_vocab_size,
        src_pad_idx,
        trg_pad_idx,
        device=device,
    ).to(device)

    trg_input_ids = trg[:, :-1]
    # (N, trg_len-1)
    out = model(x, trg_input_ids)
    # (N, trg_len-1, trg_vocab_size)
    print(out.shape)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- tiny toy vocab ---
src_vocab_size = 50
trg_vocab_size = 60
src_pad_idx = 0
trg_pad_idx = 0

# model hyperparams (paper-base 느낌, but tiny for smoke test)
model = Transformer(
    src_vocab_size=src_vocab_size,
    trg_vocab_size=trg_vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embed_size=128,
    num_layers=2,
    forward_expansion=4,
    heads=4,
    dropout=0.1,
    device=device,
    max_length=64,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --- dummy batch (variable lengths + padding) ---
src_batch = [
    torch.tensor([1, 5, 6, 4, 3, 9, 2]),
    torch.tensor([1, 8, 7, 3, 4, 5]),
]
trg_batch = [
    torch.tensor([1, 7, 4, 3, 5, 9, 2, 2]),
    torch.tensor([1, 5, 6, 2, 4, 7]),
]

src = pad_sequence(src_batch, batch_first=True, padding_value=src_pad_idx).to(device)
# (N, src_len)
trg = pad_sequence(trg_batch, batch_first=True, padding_value=trg_pad_idx).to(device)
# (N, trg_len)

trg_input = trg[:, :-1]
# (N, trg_len-1)
trg_y = trg[:, 1:]
# (N, trg_len-1)

logits = model(src, trg_input)
# (N, trg_len-1, trg_vocab_size)

logits_flat = logits.reshape(-1, logits.size(-1))
# (N*(trg_len-1), trg_vocab_size)
trg_y_flat = trg_y.reshape(-1)
# (N*(trg_len-1),)

optimizer.zero_grad()
loss = criterion(logits_flat, trg_y_flat)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

print("✅ smoke loss:", float(loss.item()))


In [ ]:
tiny_model = study_small_model(seed=23)
tiny_src = torch.tensor([[1,4,5,2], [1,6,7,2], [1,8,9,2], [1,10,11,2]])
tiny_target = torch.tensor([[1,5,4,2], [1,7,6,2], [1,9,8,2], [1,11,10,2]])
tiny_in, tiny_y = tiny_target[:, :-1], tiny_target[:, 1:]
tiny_optimizer = torch.optim.Adam(tiny_model.parameters(), lr=0.01)
tiny_loss_fn = nn.CrossEntropyLoss(ignore_index=0)
tiny_losses = []
tiny_model.train()
for step in range(120):
    tiny_optimizer.zero_grad()
    scores = tiny_model(tiny_src, tiny_in)
    loss = tiny_loss_fn(scores.flatten(0, 1), tiny_y.flatten())
    study_check(torch.isfinite(loss), "작은 데이터 loss에 NaN/Inf가 있습니다", "R12 / 학습률·mask 확인")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(tiny_model.parameters(), 1.0)
    tiny_optimizer.step()
    tiny_losses.append(loss.item())
tiny_model.eval()
with torch.no_grad():
    predicted = tiny_model(tiny_src, tiny_in).argmax(-1)
    accuracy = (predicted == tiny_y).float().mean().item()
study_check(tiny_losses[-1] < tiny_losses[0] * 0.2 and accuracy >= 0.95,
            f"작은 데이터 암기 실패: loss {tiny_losses[0]:.3f}→{tiny_losses[-1]:.3f}, acc={accuracy:.3f}",
            "R02/R09/R12 / shift·raw logits·optimizer·residual 확인")
print(f"[OK] 작은 데이터 학습: loss {tiny_losses[0]:.3f}→{tiny_losses[-1]:.3f}, token acc={accuracy:.3f}")


In [ ]:
import math
with torch.random.fork_rng():
    torch.manual_seed(31)
    print("d_k | raw variance | scaled variance | raw/scaled max probability | raw/scaled entropy")
    for width in [4, 16, 64, 256]:
        queries = torch.randn(256, 1, width)
        keys = torch.randn(256, 16, width)
        scores = (queries * keys).sum(dim=-1)
        raw_p = scores.softmax(-1)
        scaled_scores = scores / math.sqrt(width)
        scaled_p = scaled_scores.softmax(-1)
        def entropy(p):
            return -(p * p.clamp_min(1e-12).log()).sum(-1).mean().item()
        print(f"{width:3d} | {scores.var().item():10.3f} | {scaled_scores.var().item():13.3f} | "
              f"{raw_p.max(-1).values.mean().item():.3f}/{scaled_p.max(-1).values.mean().item():.3f} | "
              f"{entropy(raw_p):.3f}/{entropy(scaled_p):.3f}")


In [ ]:
import copy
position_model = study_small_model(seed=37).eval()
with_position = position_model.encoder
without_position = copy.deepcopy(with_position)
with torch.no_grad():
    without_position.position_embedding.weight.zero_()
sequence = torch.tensor([[1, 4, 7, 2]])
permutation = torch.tensor([0, 2, 1, 3])
source_mask = torch.ones(1, 1, 1, 4, dtype=torch.bool)
with torch.no_grad():
    for name, encoder in [("위치 없음", without_position), ("학습형 위치 있음", with_position)]:
        a = encoder(sequence, source_mask)
        b = encoder(sequence[:, permutation], source_mask)
        equivariance_error = (a[:, permutation] - b).abs().max().item()
        pooled_error = (a.mean(1) - b.mean(1)).abs().max().item()
        print(f"{name}: 같은 순열로 정렬한 출력 차이={equivariance_error:.6f}, 평균 벡터 차이={pooled_error:.6f}")
        if name == "위치 없음":
            study_check(equivariance_error < 1e-5 and pooled_error < 1e-5,
                        "위치 없는 encoder의 순열 성질이 맞지 않습니다", "C5/C7/C15 / 축·dropout 확인")
